# Parte 1 · Pesos del bosque, verificación y proximidad (1.4, 1.5 y 1.6)

**Autor:** Herman Echeverría · Grupo Taiwan

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por el autor.

Datos: California Housing, estandarizados como en 0.1. Los pesos del bosque se calculan con `src/bosque.py` y los de los otros cuatro modelos con `src/pesos.py`.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble._forest import _generate_sample_indices   # solo para explicar 1.4 con bootstrap

from datos import cargar_california
from pesos import pesos_ols, pesos_ridge, pesos_knn, pesos_arbol
from bosque import pesos_bosque, proximidad, verificar

X_train, X_test, y_train, y_test, _ = cargar_california()
q = X_test.iloc[:20]
n = len(X_train)

## 1.4 · Random forest

> **Pregunta 1.4 (a).** Con `RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.33, bootstrap=False, random_state=0)`, muestren que $w_i(x) = \frac{1}{T}\sum_{t=1}^{T} \frac{\mathbf 1\{x_i \text{ cae en la misma hoja que } x \text{ en el árbol } t\}}{\text{número de filas de entrenamiento en esa hoja}}$.

Sea $L_t(x)$ la hoja a la que el árbol $t$ manda a $x$. Cada árbol es un árbol de regresión, así que predice el promedio de las $y$ de entrenamiento que cayeron en esa hoja (igual que en 1.3). Con `bootstrap=False` cada árbol se ajusta con **todas** las filas de entrenamiento, así que

$$\hat y_t(x) = \sum_{i=1}^{n} \frac{\mathbf 1\{x_i \in L_t(x)\}}{|L_t(x)|}\, y_i .$$

El bosque promedia los $T$ árboles. Intercambiando las dos sumas:

$$\hat y(x) = \frac{1}{T}\sum_{t=1}^{T} \hat y_t(x) = \sum_{i=1}^{n} \underbrace{\frac{1}{T}\sum_{t=1}^{T} \frac{\mathbf 1\{x_i \in L_t(x)\}}{|L_t(x)|}}_{w_i(x)}\, y_i .$$

Es un promedio de $T$ vectores de pesos de árbol; cada uno es no negativo y suma 1, así que el promedio también. `max_features=0.33` solo cambia qué cortes se consideran, no la forma de los pesos.

In [ ]:
bosque = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.33,
                               bootstrap=False, random_state=0, n_jobs=-1).fit(X_train, y_train)
W_bosque = pesos_bosque(bosque, X_train, q)
pd.Series({"error máx. |Wy − predict|": verificar(W_bosque, y_train, bosque.predict(q)),
           "Σw mínima": W_bosque.sum(1).min(), "Σw máxima": W_bosque.sum(1).max()})